In [0]:
# ==========================================================
# PROYECTO TELCO - PRUEBAS DE CALIDAD
# Paso 1: Inspección de tablas
# ==========================================================

tablas = {
    "BRONZE": "workspace.telco_bronze.clientes_raw",
    "SILVER": "workspace.telco_silver.clientes",
    "GOLD_CLIENTES": "workspace.telco_gold.kpi_clientes_ciudad_segmento",
    "GOLD_ALTAS": "workspace.telco_gold.kpi_altas_mensuales"
}

for capa, tabla in tablas.items():

    df = spark.table(tabla)

    print("=" * 60)
    print(f"CAPA: {capa}")
    print(f"TABLA: {tabla}")
    print(f"TOTAL REGISTROS: {df.count():,}")
    print(f"COLUMNAS: {', '.join(df.columns)}")
    print("=" * 60)

In [0]:
# ==========================================================
# TESTS - CALIDAD DE DATOS TELCO
# Validaciones Bronze, Silver y Gold
# ==========================================================

from pyspark.sql import functions as F


# 1. Cargar tablas

bronze = spark.table("workspace.telco_bronze.clientes_raw")

silver = spark.table("workspace.telco_silver.clientes")

gold = spark.table(
    "workspace.telco_gold.kpi_clientes_ciudad_segmento"
)

gold_altas = spark.table(
    "workspace.telco_gold.kpi_altas_mensuales"
)


# 2. Función de validación

def validar(condicion, mensaje):

    assert condicion, f"ERROR: {mensaje}"

    print(f"✅ OK: {mensaje}")


# ==========================================================
# TEST 1: Cantidad de registros Bronze
# ==========================================================

total_bronze = bronze.count()

validar(
    total_bronze == 1000,
    "Bronze contiene los 1.000 clientes esperados"
)


# ==========================================================
# TEST 2: Consistencia Bronze vs Silver
# ==========================================================

total_silver = silver.count()

validar(
    total_silver == total_bronze,
    "Silver conserva los clientes válidos de Bronze"
)


# ==========================================================
# TEST 3: Cliente ID sin nulos
# ==========================================================

nulos_id = silver.filter(
    F.col("cliente_id").isNull()
).count()

validar(
    nulos_id == 0,
    "Silver no contiene cliente_id nulos"
)


# ==========================================================
# TEST 4: Cliente ID sin duplicados
# ==========================================================

clientes_unicos = silver.select(
    "cliente_id"
).distinct().count()

validar(
    clientes_unicos == total_silver,
    "Silver no contiene clientes duplicados"
)


# ==========================================================
# TEST 5: Fecha de alta sin nulos
# ==========================================================

nulos_fecha = silver.filter(
    F.col("fecha_alta").isNull()
).count()

validar(
    nulos_fecha == 0,
    "Silver no contiene fechas de alta nulas"
)


# ==========================================================
# TEST 6: Consistencia Silver vs Gold
# ==========================================================

resumen_gold = gold.agg(
    F.sum("total_clientes").alias("total"),
    F.sum("clientes_activos").alias("activos"),
    F.sum("clientes_inactivos").alias("inactivos")
).first()

validar(
    resumen_gold["total"] == total_silver,
    "Gold contiene el total de clientes de Silver"
)


# ==========================================================
# TEST 7: Consistencia de estados
# ==========================================================

validar(
    resumen_gold["activos"] + resumen_gold["inactivos"]
    == resumen_gold["total"],
    "Clientes activos + inactivos = total de clientes"
)


# ==========================================================
# TEST 8: Consistencia de altas mensuales
# ==========================================================

total_altas = gold_altas.agg(
    F.sum("total_altas").alias("total")
).first()["total"]

validar(
    total_altas == total_silver,
    "La suma de altas mensuales coincide con Silver"
)


print("\n" + "=" * 55)
print("✅ TODAS LAS PRUEBAS DE CALIDAD FINALIZARON CORRECTAMENTE")
print("=" * 55)

In [0]:
# ==========================================================
# TESTS - CONSISTENCIA DETALLADA GOLD
# ==========================================================

# TEST 9: Comparar clientes por ciudad y segmento

silver_grupos = silver.groupBy(
    "ciudad", "segmento"
).agg(
    F.count("*").alias("conteo_silver")
)

comparacion = silver_grupos.join(
    gold.select(
        "ciudad",
        "segmento",
        "total_clientes"
    ),
    ["ciudad", "segmento"],
    "full_outer"
)

diferencias = comparacion.filter(
    F.col("conteo_silver").isNull()
    | F.col("total_clientes").isNull()
    | (F.col("conteo_silver") != F.col("total_clientes"))
).count()

validar(
    diferencias == 0,
    "Los clientes por ciudad y segmento coinciden entre Silver y Gold"
)


# TEST 10: Verificar que Gold no tenga grupos duplicados

duplicados_gold = gold.groupBy(
    "ciudad", "segmento"
).count().filter(
    F.col("count") > 1
).count()

validar(
    duplicados_gold == 0,
    "Gold no contiene ciudades y segmentos duplicados"
)


# TEST 11: Comparar altas mensuales acumuladas contra Gold

altas_grupos = gold_altas.groupBy(
    "ciudad", "segmento"
).agg(
    F.sum("total_altas").alias("total_altas")
)

comparacion_altas = altas_grupos.join(
    gold.select(
        "ciudad",
        "segmento",
        "total_clientes"
    ),
    ["ciudad", "segmento"],
    "full_outer"
)

diferencias_altas = comparacion_altas.filter(
    F.col("total_altas").isNull()
    | F.col("total_clientes").isNull()
    | (F.col("total_altas") != F.col("total_clientes"))
).count()

validar(
    diferencias_altas == 0,
    "Las altas acumuladas coinciden con Gold por ciudad y segmento"
)


# TEST 12: Validar el porcentaje de clientes activos

porcentajes_incorrectos = gold.filter(
    (F.col("total_clientes") <= 0)
    | F.col("porcentaje_activos").isNull()
    | (
        F.abs(
            F.col("porcentaje_activos")
            - (
                F.col("clientes_activos") * 100.0
                / F.col("total_clientes")
            )
        ) > 0.11
    )
).count()

validar(
    porcentajes_incorrectos == 0,
    "Los porcentajes de clientes activos están correctamente calculados"
)


print("\n✅ PRUEBAS DETALLADAS DE GOLD FINALIZADAS")